# Extracting the DP2 VisitDetector Table 

This notebook provides a recipe for extracting the information from the Rubin DP2 VisitDetector table that is needed by LightCurveLynx to perform simulations and save it in parquet format. It is meant to be run on the [Rubin Science Platform](https://data.lsst.cloud/).

In [ ]:
from lsst.rsp import get_tap_service

service = get_tap_service("tap")

We start by retrieving and displaying the meta data for the dp2.VisitDetector table.

In [ ]:
my_adql_query = (
    "SELECT column_name, datatype, description, unit "
    + "FROM TAP_SCHEMA.columns "
    + "WHERE table_name = 'dp2.VisitDetector'"
)
adql_results = service.search(my_adql_query)
adql_results.to_table().to_pandas()

Next we extract a subset of the columns into a AstroPy table. This command takes a while to run on the RSP (>10 minutes).

In [ ]:
col_list = [
    "expMidptMJD",  # Midpoint time for exposure
    "ra",  # Right Ascension of Ccd center (degrees)
    "dec",  # Declination of Ccd center  (degrees)
    "band",  # Name of the filter used.
    "skyRotation",  #  Sky rotation angle (degrees)
    "magLim ",  # 5-sigma limiting magnitude
    "expTime",  # The exposure time (seconds)
    "seeing",  # Mean measured FWHM of the PSF
    "skyBg",  # Average sky background (adu)
    "skyNoise",  # RMS noise of the sky background (adu)
    "pixelScale",  # pixel scale (arcsec/pixel)
    "xSize",  # Number of pixels in x direction
    "ySize",  # Number of pixels in y direction
    "zeroPoint",  # Zero-point for the Ccd (mag)
]
adql_query = "SELECT " + ", ".join(col_list) + " FROM dp2.VisitDetector"

# Run the job asynchronously
job = service.submit_job(adql_query)
job.run()
job.wait(phases=["COMPLETED", "ERROR"])
job.raise_if_error()

# Fetch the result as an astropy table.
results_table = job.fetch_result().to_table()
job.delete()

In [ ]:
pandas_table = results_table.to_pandas()
pandas_table.head()

Finally we save the data to a parquet file (>200MB).

In [ ]:
obstable_filename = "dp2_visitdetector.parquet"
pandas_table.to_parquet(obstable_filename)